In [2]:
from datasets import load_dataset

data=load_dataset("glue","rte")

In [3]:
data

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 2490
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 277
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3000
    })
})

In [4]:
data["train"][0]


{'sentence1': 'No Weapons of Mass Destruction Found in Iraq Yet.',
 'sentence2': 'Weapons of Mass Destruction Found in Iraq.',
 'label': 1,
 'idx': 0}

In [5]:
from transformers import AutoTokenizer,AutoModelForSequenceClassification,TrainingArguments,Trainer



In [6]:
tokenizer=AutoTokenizer.from_pretrained("bert-base-uncased")

C:\Users\Lenovo\AppData\Roaming\Python\Python311\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [7]:
def tokenize_function(data):
    return tokenizer(data["sentence1"], data["sentence2"], truncation=True)

tokenized_data = data.map(tokenize_function, batched=True)

Map:   0%|          | 0/277 [00:00<?, ? examples/s]

In [8]:
label2id={x["label"]: x["label"] for x in data["train"]}
label2id

{1: 1, 0: 0}

In [9]:
label2id={"Yes" : 1, "No": 0} # for RTE dataset
id2label={ v:k for k,v in label2id.items()} # vice versa of label2id
label2id

{'Yes': 1, 'No': 0}

In [10]:
from transformers import AutoConfig

config=AutoConfig.from_pretrained("bert-base-uncased",label2id=label2id,id2label=id2label)

In [11]:
model=AutoModelForSequenceClassification.from_pretrained("bert-base-uncased",config=config)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
training_args = TrainingArguments(
    output_dir="result",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    logging_steps= 150
)

In [13]:
from sklearn.metrics import accuracy_score,f1_score

def compute_metrics(eval_pred):
    predictions,labels = eval_pred
    pred= predictions.argmax(axis=1)
    accuracy = accuracy_score(labels, pred)
    f1=f1_score(labels, pred)
    return {"accuracy": accuracy,
            "f1": f1}


In [14]:
trainer=Trainer(
    
    model=model,
    args=training_args,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["validation"],
    compute_metrics=compute_metrics,
    tokenizer=tokenizer,
)

In [ ]:
trainer.train()

  0%|          | 0/468 [00:00<?, ?it/s]

c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [ ]:
model.save("model")

In [ ]:
from transformers import pipeline

alignment_model = pipeline("text-similarity", model="model", tokenizer=tokenizer)

alignment_model("The cat is on the roof.", "The dog is on the roof.")